# Fase 04: Transformación de datos

Este notebook implementa la **Fase 04** del proyecto de Modelado Avanzado de Base de Datos.

El objetivo es tomar el dataset reconstruido en la capa **bronze** y convertirlo en un dataset transformado, consistente y útil para análisis, dejando el resultado en la capa **silver**.

En esta fase se aplican las transformaciones obligatorias solicitadas en el caso de estudio:

1. Normalización de nombres de columnas.
2. Conversión de fechas a `timestamp`.
3. Cálculo de duración del viaje en minutos.
4. Cálculo de velocidad promedio estimada.
5. Normalización de importes monetarios.
6. Identificación de valores negativos inválidos.
7. Generación de identificador único `trip_id`.
8. Eliminación de duplicados técnicos.
9. Enriquecimiento con `year` y `month`.
10. Creación o validación de `service_type`.
11. Creación o validación de `source_file`.
12. Creación o validación de `ingestion_timestamp`.
13. Creación de `quality_status`.

Además, se generan los campos derivados mínimos:

- `trip_duration_minutes`
- `average_speed_mph`
- `fare_per_mile`
- `tip_percentage`
- `is_airport_trip`
- `is_suspicious_trip`
- `processing_date`


## 1. Preparación del entorno

Esta sección configura Python, Hadoop y Spark para trabajar correctamente en Windows.

Se define explícitamente el Python que debe usar PySpark mediante `sys.executable`. Esto evita el error típico de Windows donde Spark intenta buscar `python3` y no lo encuentra.


In [1]:
# Preparar Python, Hadoop y Spark para trabajar en Windows.

import os
import sys
import json
from pathlib import Path
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Python usado por Jupyter:")
print(sys.executable)
print("PYSPARK_PYTHON:")
print(os.environ["PYSPARK_PYTHON"])
print("Existe winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("Existe hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

Python usado por Jupyter:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
PYSPARK_PYTHON:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
Existe winutils: False
Existe hadoop.dll: False


In [2]:
# Importar librerías de Spark.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType,
    BooleanType
)


In [3]:
# Crear sesión Spark.
# Se usan configuraciones moderadas para ejecución local en Windows.

spark = (
    SparkSession.builder
    .appName("Fase_04_Transformacion_Datos")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.io.native.lib.available", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente")
print("Versión Spark:", spark.version)

Spark iniciado correctamente
Versión Spark: 4.1.2


## 2. Rutas del proyecto y `process_id`

Se definen las rutas principales de la arquitectura Lakehouse:

- `bronze`: contiene el dataset reconstruido en la fase anterior.
- `silver`: recibirá el dataset transformado.
- `quarantine`: recibirá referencias de registros inválidos detectados en transformación.
- `audit`: conserva evidencia técnica del procesamiento.

También se genera un `process_id`, que identifica de forma única esta ejecución del pipeline.


In [4]:
# Definir rutas principales.

BASE_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_PATH / "data"
BRONZE_PATH = DATA_PATH / "bronze"
SILVER_PATH = DATA_PATH / "silver"
QUARANTINE_PATH = DATA_PATH / "quarantine"
AUDIT_PATH = DATA_PATH / "audit"

PROCESS_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

SILVER_PATH.mkdir(parents=True, exist_ok=True)
QUARANTINE_PATH.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("PROCESS_ID:", PROCESS_ID)

BASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
BRONZE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze
SILVER_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver
QUARANTINE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
PROCESS_ID: 20260617_181025


In [5]:
# Función auxiliar para convertir rutas de Windows a formato compatible con Spark.

def spark_path(path):
    return str(Path(path).resolve()).replace("\\", "/")

print("Ejemplo ruta Spark:")
print(spark_path(SILVER_PATH))

Ejemplo ruta Spark:
C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/silver


## 3. Localización del dataset Bronze reconstruido

La Fase 04 necesita leer el dataset reconstruido por las fases anteriores.

El notebook busca automáticamente la carpeta más reciente dentro de `data/bronze` que contenga el dataset canónico reconstruido. Si existen varias ejecuciones, se toma la más reciente según la fecha de modificación.


In [6]:
# Buscar la carpeta bronze más reciente que contenga trips reconstruidos.

bronze_candidates = []

if BRONZE_PATH.exists():
    for path in BRONZE_PATH.glob("trips_canonical_*"):
        if path.is_dir():
            bronze_candidates.append(path)

# También se considera una posible carpeta de archivos recuperados si existe.
if BRONZE_PATH.exists():
    for path in BRONZE_PATH.glob("recovered_files_*"):
        if path.is_dir():
            bronze_candidates.append(path)

if len(bronze_candidates) == 0:
    raise FileNotFoundError(
        "No se encontró un dataset reconstruido en data/bronze. "
        "Ejecuta primero la Fase 02 y, si aplica, la Fase 03."
    )

latest_bronze_path = max(bronze_candidates, key=lambda p: p.stat().st_mtime)

print("Dataset Bronze seleccionado:")
print(latest_bronze_path)

Dataset Bronze seleccionado:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze\trips_canonical_fase2_20260617_170931


## 4. Lectura del dataset reconstruido

Se lee el dataset desde Bronze usando Spark.

Esta lectura se realiza directamente desde Parquet para mantener el procesamiento distribuido y evitar convertir el trabajo a Pandas, tal como exige el caso de estudio.


In [7]:
# Leer dataset reconstruido desde Bronze.
# Bronze está guardado por servicio: service_type=yellow, service_type=green, service_type=fhvhv.
# Por eso se leen recursivamente las subcarpetas.

bronze_trips = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(spark_path(latest_bronze_path))
)

print("Dataset Bronze cargado correctamente.")
print("Total de registros Bronze:", bronze_trips.count())
print("Total de columnas Bronze:", len(bronze_trips.columns))
bronze_trips.printSchema()
bronze_trips.show(5, truncate=False)

Dataset Bronze cargado correctamente.
Total de registros Bronze: 44502927
Total de columnas Bronze: 23
root
 |-- trip_id: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- vendor_id: long (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- source_

## 5. Normalización de nombres de columnas

Aunque el dataset ya viene reconstruido, se normalizan los nombres de columnas para garantizar consistencia.

La normalización aplicada consiste en:

- Quitar espacios al inicio y final.
- Convertir nombres a minúsculas.
- Reemplazar espacios, guiones y caracteres especiales por `_`.

Esto evita errores posteriores al consultar o transformar columnas.


In [8]:
# Normalizar nombres de columnas.

def normalize_column_name(column_name):
    normalized = column_name.strip().lower()
    normalized = normalized.replace(" ", "_")
    normalized = normalized.replace("-", "_")
    normalized = normalized.replace(".", "_")
    normalized = normalized.replace("/", "_")
    normalized = normalized.replace("(", "")
    normalized = normalized.replace(")", "")
    return normalized

normalized_columns = [F.col(col).alias(normalize_column_name(col)) for col in bronze_trips.columns]

trips_normalized = bronze_trips.select(*normalized_columns)

print("Columnas normalizadas:")
print(trips_normalized.columns)

Columnas normalizadas:
['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'quality_status']


## 6. Asegurar columnas mínimas del esquema canónico

Antes de transformar, se asegura que existan las columnas necesarias para calcular las métricas solicitadas.

Si una columna no existe, se crea con valor `NULL`. Esto mantiene un esquema estable y evita errores al procesar diferentes servicios como Yellow, Green y FHVHV.


In [9]:
# Columnas mínimas necesarias para transformación.

required_columns = [
    "trip_id",
    "service_type",
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "fare_amount",
    "extra_amount",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "year",
    "month",
    "source_file",
    "ingestion_timestamp",
    "quality_status"
]

trips_prepared = trips_normalized

for column_name in required_columns:
    if column_name not in trips_prepared.columns:
        trips_prepared = trips_prepared.withColumn(column_name, F.lit(None))
        print("Columna creada con NULL controlado:", column_name)

print("Validación de columnas mínimas completada.")

Validación de columnas mínimas completada.


## 7. Conversión de tipos y normalización monetaria

En esta sección se convierten los campos a tipos adecuados:

- Fechas a `timestamp`.
- Campos numéricos a `double` o `int`.
- Campos de texto a `string`.

También se normalizan los importes monetarios manteniéndolos como valores numéricos. Los valores negativos no se eliminan aquí; primero se marcan como inválidos para mantener trazabilidad y luego se separan.


In [10]:
# Conversión de tipos de datos.

trips_typed = (
    trips_prepared
    .withColumn("pickup_datetime", F.to_timestamp(F.col("pickup_datetime")))
    .withColumn("dropoff_datetime", F.to_timestamp(F.col("dropoff_datetime")))
    .withColumn("ingestion_timestamp", F.to_timestamp(F.col("ingestion_timestamp")))
    .withColumn("service_type", F.col("service_type").cast("string"))
    .withColumn("source_file", F.col("source_file").cast("string"))
    .withColumn("quality_status", F.col("quality_status").cast("string"))
    .withColumn("vendor_id", F.col("vendor_id").cast("string"))
    .withColumn("payment_type", F.col("payment_type").cast("string"))
    .withColumn("passenger_count", F.col("passenger_count").cast("double"))
    .withColumn("trip_distance", F.col("trip_distance").cast("double"))
    .withColumn("pickup_location_id", F.col("pickup_location_id").cast("int"))
    .withColumn("dropoff_location_id", F.col("dropoff_location_id").cast("int"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("extra_amount", F.col("extra_amount").cast("double"))
    .withColumn("mta_tax", F.col("mta_tax").cast("double"))
    .withColumn("tip_amount", F.col("tip_amount").cast("double"))
    .withColumn("tolls_amount", F.col("tolls_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
    .withColumn("airport_fee", F.col("airport_fee").cast("double"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
)

print("Tipos convertidos correctamente.")
trips_typed.printSchema()

Tipos convertidos correctamente.
root
 |-- trip_id: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: tim

## 8. Enriquecimiento técnico obligatorio

Se asegura la existencia de campos técnicos que permiten auditar el dataset:

- `service_type`: identifica el tipo de servicio.
- `source_file`: identifica el archivo origen.
- `ingestion_timestamp`: identifica cuándo ingresó el registro.
- `processing_date`: identifica la fecha de procesamiento de esta fase.
- `year` y `month`: permiten particionar y analizar por periodo.

Cuando `year` o `month` no están disponibles, se calculan desde `pickup_datetime`.


In [11]:
# Enriquecimiento técnico.

trips_enriched = (
    trips_typed
    .withColumn(
        "year",
        F.when(F.col("year").isNotNull(), F.col("year"))
         .otherwise(F.year(F.col("pickup_datetime")))
    )
    .withColumn(
        "month",
        F.when(F.col("month").isNotNull(), F.col("month"))
         .otherwise(F.month(F.col("pickup_datetime")))
    )
    .withColumn(
        "ingestion_timestamp",
        F.when(F.col("ingestion_timestamp").isNotNull(), F.col("ingestion_timestamp"))
         .otherwise(F.current_timestamp())
    )
    .withColumn("processing_date", F.current_date())
    .withColumn(
        "source_file",
        F.when(F.col("source_file").isNotNull(), F.col("source_file"))
         .otherwise(F.lit("UNKNOWN_SOURCE_FILE"))
    )
    .withColumn(
        "service_type",
        F.when(F.col("service_type").isNotNull(), F.lower(F.col("service_type")))
         .otherwise(F.lit("unknown"))
    )
)

print("Campos técnicos enriquecidos.")
trips_enriched.select("service_type", "year", "month", "source_file", "ingestion_timestamp", "processing_date").show(10, truncate=False)

Campos técnicos enriquecidos.
+------------+----+-----+------------------------------+--------------------------+---------------+
|service_type|year|month|source_file                   |ingestion_timestamp       |processing_date|
+------------+----+-----+------------------------------+--------------------------+---------------+
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:16:20.261196|2026-06-17     |
|fhvhv       |2023|1    |fhvhv_tripdata_2023-01.parquet|2026-06-17 17:

## 9. Generación de identificador único `trip_id`

El campo `trip_id` identifica cada viaje de forma única.

Si el registro ya trae `trip_id`, se conserva. Si no existe, se genera mediante un hash SHA-256 a partir de campos relevantes del viaje:

- Tipo de servicio.
- Fecha de recogida.
- Fecha de llegada.
- Zona de origen.
- Zona de destino.
- Distancia.
- Total pagado.
- Archivo fuente.

Esto permite construir un identificador técnico reproducible para detectar duplicados.


In [12]:
# Generar trip_id si está ausente.

trips_with_id = trips_enriched.withColumn(
    "generated_trip_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("service_type"), F.lit("")),
            F.coalesce(F.col("pickup_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("dropoff_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("pickup_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("dropoff_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
            F.coalesce(F.col("source_file"), F.lit(""))
        ),
        256
    )
)

trips_with_id = (
    trips_with_id
    .withColumn(
        "trip_id",
        F.when(F.col("trip_id").isNotNull(), F.col("trip_id").cast("string"))
         .otherwise(F.col("generated_trip_id"))
    )
    .drop("generated_trip_id")
)

print("trip_id generado o validado correctamente.")
trips_with_id.select("trip_id", "service_type", "source_file").show(5, truncate=False)

trip_id generado o validado correctamente.
+----------------------------------------------------------------+------------+------------------------------+
|trip_id                                                         |service_type|source_file                   |
+----------------------------------------------------------------+------------+------------------------------+
|cdcf7d1d257a4ca5898c0a398c1e78a88ea8623746bdf3e353f11a618bfb84fb|fhvhv       |fhvhv_tripdata_2023-01.parquet|
|ab3a6cd672505c647319d31ee34f43ee8616e864b6bdca5bbe656703626a56e4|fhvhv       |fhvhv_tripdata_2023-01.parquet|
|1eb4d98f5769f667fa788d803958e3c3e1716cbb4bcd2f06f8e2de0335668659|fhvhv       |fhvhv_tripdata_2023-01.parquet|
|da76982ce194737ee41626302b3e5cdf755e84e232795bc6aa064ac64b3aec62|fhvhv       |fhvhv_tripdata_2023-01.parquet|
|6826782afbb93b6ad52b7fce0d3825151e4e12de7a1affccc135335a63f139e1|fhvhv       |fhvhv_tripdata_2023-01.parquet|
+----------------------------------------------------------------+---

## 10. Cálculo de campos derivados

Se calculan los campos mínimos solicitados por la Fase 04:

- `trip_duration_minutes`: duración del viaje en minutos.
- `average_speed_mph`: velocidad promedio en millas por hora.
- `fare_per_mile`: tarifa por milla.
- `tip_percentage`: porcentaje de propina sobre la tarifa.
- `is_airport_trip`: indica si el viaje involucra aeropuerto.
- `is_suspicious_trip`: indica si el viaje cumple alguna regla sospechosa.
- `processing_date`: fecha de procesamiento.


In [13]:
# Calcular duración, velocidad, tarifa por milla y porcentaje de propina.

trips_derived = (
    trips_with_id
    .withColumn(
        "trip_duration_minutes",
        (F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / F.lit(60.0)
    )
    .withColumn(
        "average_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") / (F.col("trip_duration_minutes") / F.lit(60.0))
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.col("fare_amount") / F.col("trip_distance")
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "tip_percentage",
        F.when(
            F.col("fare_amount") > 0,
            (F.col("tip_amount") / F.col("fare_amount")) * F.lit(100.0)
        ).otherwise(F.lit(None).cast("double"))
    )
)

print("Campos derivados calculados.")
trips_derived.select(
    "trip_duration_minutes",
    "average_speed_mph",
    "fare_per_mile",
    "tip_percentage"
).show(10, truncate=False)

Campos derivados calculados.
+---------------------+------------------+------------------+-----------------+
|trip_duration_minutes|average_speed_mph |fare_per_mile     |tip_percentage   |
+---------------------+------------------+------------------+-----------------+
|28.483333333333334   |1.9801053247513165|27.606382978723406|20.11560693641618|
|34.483333333333334   |4.837119381343644 |21.63309352517986 |0.0              |
|17.45                |30.29226361031519 |2.7661748013620886|0.0              |
|7.183333333333334    |5.596287703016241 |20.597014925373134|0.0              |
|12.066666666666666   |21.77900552486188 |4.678082191780821 |0.0              |
|20.816666666666666   |14.18382706164932 |3.7167242430400322|0.0              |
|20.633333333333333   |16.04297253634895 |4.6692042776871485|0.0              |
|7.883333333333334    |14.384778012684988|7.677248677248677 |0.0              |
|11.066666666666666   |14.367469879518072|4.90566037735849  |0.0              |
|24.6833333

## 11. Identificación de viajes de aeropuerto

El campo `is_airport_trip` se genera usando una regla técnica simple basada en zonas de taxi de NYC.

La zona `132` corresponde comúnmente a JFK y la zona `138` corresponde comúnmente a LaGuardia. Si el origen o destino coincide con esas zonas, el viaje se marca como viaje de aeropuerto.

Esta regla permite enriquecer el dataset para análisis posterior.


In [14]:
# Crear indicador de viaje de aeropuerto.

airport_location_ids = [132, 138]

trips_derived = trips_derived.withColumn(
    "is_airport_trip",
    F.when(
        F.col("pickup_location_id").isin(airport_location_ids) |
        F.col("dropoff_location_id").isin(airport_location_ids) |
        (F.coalesce(F.col("airport_fee"), F.lit(0.0)) > 0),
        F.lit(True)
    ).otherwise(F.lit(False))
)

print("Indicador is_airport_trip creado.")
trips_derived.select("pickup_location_id", "dropoff_location_id", "airport_fee", "is_airport_trip").show(10, truncate=False)

Indicador is_airport_trip creado.
+------------------+-------------------+-----------+---------------+
|pickup_location_id|dropoff_location_id|airport_fee|is_airport_trip|
+------------------+-------------------+-----------+---------------+
|48                |68                 |NULL       |false          |
|246               |163                |NULL       |false          |
|9                 |129                |NULL       |false          |
|129               |129                |NULL       |false          |
|129               |92                 |NULL       |false          |
|130               |38                 |NULL       |false          |
|38                |10                 |NULL       |false          |
|90                |231                |NULL       |false          |
|125               |246                |NULL       |false          |
|68                |231                |NULL       |false          |
+------------------+-------------------+-----------+---------------+


## 12. Reglas de viaje sospechoso

Se marca un viaje como sospechoso cuando cumple una o más condiciones solicitadas por el caso de estudio:

- `trip_distance <= 0`
- `total_amount <= 0`
- `fare_amount < 0`
- `trip_duration_minutes <= 0`
- `trip_duration_minutes > 480`
- `average_speed_mph > 100`
- `tip_percentage > 100`
- `pickup_datetime > dropoff_datetime`
- `pickup_datetime` está en el futuro

Esta marca no elimina el registro inmediatamente. Primero se registra la condición para conservar trazabilidad.


In [15]:
# Crear reglas de sospecha.

current_timestamp = F.current_timestamp()

trips_flagged = (
    trips_derived
    .withColumn("rule_invalid_distance", F.col("trip_distance") <= 0)
    .withColumn("rule_invalid_total_amount", F.col("total_amount") <= 0)
    .withColumn("rule_negative_fare", F.col("fare_amount") < 0)
    .withColumn("rule_invalid_duration", F.col("trip_duration_minutes") <= 0)
    .withColumn("rule_excessive_duration", F.col("trip_duration_minutes") > 480)
    .withColumn("rule_unreal_speed", F.col("average_speed_mph") > 100)
    .withColumn("rule_excessive_tip", F.col("tip_percentage") > 100)
    .withColumn("rule_pickup_after_dropoff", F.col("pickup_datetime") > F.col("dropoff_datetime"))
    .withColumn("rule_future_pickup", F.col("pickup_datetime") > current_timestamp)
)

trips_flagged = trips_flagged.withColumn(
    "is_suspicious_trip",
    F.coalesce(F.col("rule_invalid_distance"), F.lit(False)) |
    F.coalesce(F.col("rule_invalid_total_amount"), F.lit(False)) |
    F.coalesce(F.col("rule_negative_fare"), F.lit(False)) |
    F.coalesce(F.col("rule_invalid_duration"), F.lit(False)) |
    F.coalesce(F.col("rule_excessive_duration"), F.lit(False)) |
    F.coalesce(F.col("rule_unreal_speed"), F.lit(False)) |
    F.coalesce(F.col("rule_excessive_tip"), F.lit(False)) |
    F.coalesce(F.col("rule_pickup_after_dropoff"), F.lit(False)) |
    F.coalesce(F.col("rule_future_pickup"), F.lit(False))
)

print("Indicador is_suspicious_trip generado.")
trips_flagged.groupBy("is_suspicious_trip").count().show()

Indicador is_suspicious_trip generado.
+------------------+--------+
|is_suspicious_trip|   count|
+------------------+--------+
|              true|  597351|
|             false|43905576|
+------------------+--------+



## 13. Separación de registros rechazados por reglas de transformación

La Fase 04 pide convertir valores negativos inválidos en registros rechazados.

En este notebook se separan como rechazados los registros que incumplen reglas críticas de transformación, por ejemplo:

- Fechas nulas.
- Fecha de recogida mayor que fecha de llegada.
- Distancia menor o igual a cero.
- Montos negativos o inválidos.
- Duración inválida.
- Velocidad irreal.

Los registros rechazados no se eliminan sin explicación; se guardan con regla, causa técnica y causa de negocio.


In [16]:
# Definir condición crítica de rechazo para transformación.

critical_rejection_condition = (
    F.col("pickup_datetime").isNull() |
    F.col("dropoff_datetime").isNull() |
    F.col("trip_distance").isNull() |
    F.col("fare_amount").isNull() |
    F.col("total_amount").isNull() |
    (F.col("trip_distance") <= 0) |
    (F.col("total_amount") <= 0) |
    (F.col("fare_amount") < 0) |
    (F.col("trip_duration_minutes") <= 0) |
    (F.col("trip_duration_minutes") > 480) |
    (F.col("average_speed_mph") > 100) |
    (F.col("tip_percentage") > 100) |
    (F.col("pickup_datetime") > F.col("dropoff_datetime")) |
    (F.col("pickup_datetime") > F.current_timestamp())
)

trips_rejected_stage4 = trips_flagged.filter(critical_rejection_condition)
trips_candidate_valid = trips_flagged.filter(~critical_rejection_condition)

print("Registros rechazados en Fase 04:", trips_rejected_stage4.count())
print("Registros candidatos válidos:", trips_candidate_valid.count())

Registros rechazados en Fase 04: 597351
Registros candidatos válidos: 43903215


## 14. Tabla de registros rechazados de transformación

Se crea una tabla de rechazados con trazabilidad.

Aunque la tabla final `quality_rejected_records` se consolidará en Fase 05, esta fase deja una salida intermedia con el motivo técnico del rechazo.


In [17]:
# Crear registros rechazados con causa principal.

rejected_transform_records = (
    trips_rejected_stage4
    .withColumn("process_id", F.lit(PROCESS_ID))
    .withColumn("rejection_stage", F.lit("FASE_04_TRANSFORMACION"))
    .withColumn(
        "rejection_rule",
        F.when(F.col("pickup_datetime").isNull(), F.lit("NULL_PICKUP_DATETIME"))
         .when(F.col("dropoff_datetime").isNull(), F.lit("NULL_DROPOFF_DATETIME"))
         .when(F.col("trip_distance").isNull(), F.lit("NULL_TRIP_DISTANCE"))
         .when(F.col("fare_amount").isNull(), F.lit("NULL_FARE_AMOUNT"))
         .when(F.col("total_amount").isNull(), F.lit("NULL_TOTAL_AMOUNT"))
         .when(F.col("trip_distance") <= 0, F.lit("INVALID_TRIP_DISTANCE"))
         .when(F.col("total_amount") <= 0, F.lit("INVALID_TOTAL_AMOUNT"))
         .when(F.col("fare_amount") < 0, F.lit("NEGATIVE_FARE_AMOUNT"))
         .when(F.col("trip_duration_minutes") <= 0, F.lit("INVALID_TRIP_DURATION"))
         .when(F.col("trip_duration_minutes") > 480, F.lit("EXCESSIVE_TRIP_DURATION"))
         .when(F.col("average_speed_mph") > 100, F.lit("UNREALISTIC_SPEED"))
         .when(F.col("tip_percentage") > 100, F.lit("EXCESSIVE_TIP_PERCENTAGE"))
         .when(F.col("pickup_datetime") > F.col("dropoff_datetime"), F.lit("PICKUP_AFTER_DROPOFF"))
         .when(F.col("pickup_datetime") > F.current_timestamp(), F.lit("FUTURE_PICKUP_DATETIME"))
         .otherwise(F.lit("UNKNOWN_TRANSFORMATION_REJECTION"))
    )
    .withColumn(
        "technical_reason",
        F.concat(
            F.lit("Registro rechazado durante transformación por regla: "),
            F.col("rejection_rule")
        )
    )
    .withColumn(
        "business_reason",
        F.lit("El viaje no cumple condiciones mínimas para análisis confiable.")
    )
    .withColumn("rejected_at", F.current_timestamp())
)

rejected_transform_records.select(
    "process_id", "trip_id", "service_type", "source_file", "rejection_stage",
    "rejection_rule", "technical_reason", "business_reason", "rejected_at"
).show(20, truncate=False)

+---------------+----------------------------------------------------------------+------------+------------------------------+----------------------+------------------------+-----------------------------------------------------------------------------+---------------------------------------------------------------+--------------------------+
|process_id     |trip_id                                                         |service_type|source_file                   |rejection_stage       |rejection_rule          |technical_reason                                                             |business_reason                                                |rejected_at               |
+---------------+----------------------------------------------------------------+------------+------------------------------+----------------------+------------------------+-----------------------------------------------------------------------------+------------------------------------------------------------

## 15. Eliminación de duplicados técnicos

Se eliminan duplicados técnicos usando `trip_id`.

La estrategia aplicada conserva el primer registro por `trip_id`. Esta decisión permite mantener idempotencia y evita duplicar viajes cuando el pipeline se ejecuta más de una vez.


In [18]:
# Eliminar duplicados técnicos por trip_id de forma segura.
# Se conserva el primer registro por trip_id para mantener idempotencia.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Iniciando eliminación de duplicados técnicos...")

# Reducir particiones para evitar sobrecargar Spark en Windows
trips_candidate_valid = trips_candidate_valid.repartition(8, "trip_id")

before_dedup_count = trips_candidate_valid.count()

# Ventana para identificar duplicados sin hacer collect pesado
window_trip = Window.partitionBy("trip_id").orderBy(F.col("ingestion_timestamp").asc())

trips_with_row_number = (
    trips_candidate_valid
    .withColumn("rn_dedup", F.row_number().over(window_trip))
)

# Registros duplicados técnicos
duplicate_records = (
    trips_with_row_number
    .filter(F.col("rn_dedup") > 1)
    .drop("rn_dedup")
)

# Registros limpios sin duplicados
trips_clean_stage4 = (
    trips_with_row_number
    .filter(F.col("rn_dedup") == 1)
    .drop("rn_dedup")
)

after_dedup_count = trips_clean_stage4.count()
duplicate_count = before_dedup_count - after_dedup_count

print("Registros antes de deduplicar:", before_dedup_count)
print("Registros después de deduplicar:", after_dedup_count)
print("Duplicados técnicos eliminados:", duplicate_count)

Iniciando eliminación de duplicados técnicos...
Registros antes de deduplicar: 43903215
Registros después de deduplicar: 43891160
Duplicados técnicos eliminados: 12055


## 16. Estado de calidad después de transformación

A los registros que superan las reglas críticas y la deduplicación se les asigna estado `TRANSFORMED_VALID`.

Si un registro no fue rechazado pero fue marcado como sospechoso, se mantiene como `TRANSFORMED_SUSPICIOUS`. Esto permite conservarlo para análisis posterior, pero con advertencia de calidad.


In [19]:
# Asignar estado de calidad final de Fase 04.

trips_transformed = (
    trips_clean_stage4
    .withColumn(
        "quality_status",
        F.when(F.col("is_suspicious_trip") == True, F.lit("TRANSFORMED_SUSPICIOUS"))
         .otherwise(F.lit("TRANSFORMED_VALID"))
    )
)

print("Estados de calidad generados:")
trips_transformed.groupBy("quality_status").count().show()

Estados de calidad generados:
+-----------------+--------+
|   quality_status|   count|
+-----------------+--------+
|TRANSFORMED_VALID|43891160|
+-----------------+--------+



## 17. Selección ordenada de columnas de salida Silver

Se ordenan las columnas principales del dataset transformado.

La salida Silver conserva campos canónicos, campos derivados, trazabilidad y estado de calidad.


In [20]:
# Seleccionar columnas finales para Silver.

silver_columns = [
    "trip_id",
    "service_type",
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "passenger_count",
    "trip_distance",
    "average_speed_mph",
    "fare_per_mile",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "fare_amount",
    "extra_amount",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "tip_percentage",
    "congestion_surcharge",
    "airport_fee",
    "is_airport_trip",
    "is_suspicious_trip",
    "year",
    "month",
    "source_file",
    "ingestion_timestamp",
    "processing_date",
    "quality_status"
]

trips_silver = trips_transformed.select(*silver_columns)

print("Dataset Silver preparado.")
print("Columnas Silver:", len(trips_silver.columns))
trips_silver.printSchema()

# No se ejecuta count() ni show() aqui porque fuerzan el procesamiento completo
# del linaje y pueden tumbar la JVM de Spark en ejecucion local Windows.
print("Vista previa omitida para mantener estable Spark antes del guardado.")

Dataset Silver preparado.
Columnas Silver: 30
root
 |-- trip_id: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- average_speed_mph: double (nullable = true)
 |-- fare_per_mile: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- congestion_surcharge: double (n

## 18. Guardado del dataset transformado en Silver

Se guarda el dataset transformado en la capa `data/silver`.

La escritura se realiza particionada por `service_type`, `year` y `month`. Esto mejora el acceso posterior a los datos y ayuda a aplicar partition pruning en consultas analíticas.


In [21]:
# Guardar dataset transformado en Silver.

silver_output_path = SILVER_PATH / f"trips_transformed_{PROCESS_ID}"

(
    trips_silver
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("service_type", "year", "month")
    .parquet(spark_path(silver_output_path))
)

print("Dataset transformado guardado en Silver:")
print(silver_output_path)

Dataset transformado guardado en Silver:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver\trips_transformed_20260617_181025


## 19. Guardado de rechazados de transformación

Los registros rechazados durante la Fase 04 se guardan en `data/quarantine`.

Esto permite mantener evidencia de los registros que no pasaron las reglas mínimas de transformación.


In [22]:
# Guardar rechazados de transformación.

rejected_output_path = QUARANTINE_PATH / f"rejected_records_phase04_{PROCESS_ID}"

rejected_output_columns = [
    "process_id",
    "trip_id",
    "service_type",
    "source_file",
    "rejection_stage",
    "rejection_rule",
    "technical_reason",
    "business_reason",
    "rejected_at"
]

(
    rejected_transform_records
    .select(*rejected_output_columns)
    .write
    .mode("overwrite")
    .parquet(spark_path(rejected_output_path))
)

print("Registros rechazados de Fase 04 guardados en:")
print(rejected_output_path)

Registros rechazados de Fase 04 guardados en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\rejected_records_phase04_20260617_181025


## 20. Resumen de auditoría de transformación

Se crea un resumen técnico de la Fase 04 con los principales conteos:

- Registros de entrada desde Bronze.
- Registros rechazados.
- Duplicados detectados.
- Registros finales en Silver.
- Registros sospechosos.

Este resumen se guarda en la capa `audit`.


In [23]:
# Crear resumen de auditoría de Fase 04.

input_records = bronze_trips.count()
rejected_records = trips_rejected_stage4.count()
silver_saved = spark.read.parquet(spark_path(silver_output_path))
silver_records = silver_saved.count()
suspicious_records = silver_saved.filter(F.col("is_suspicious_trip") == True).count()

phase04_audit_data = [{
    "process_id": PROCESS_ID,
    "phase": "FASE_04_TRANSFORMACION_DATOS",
    "bronze_input_records": int(input_records),
    "rejected_records": int(rejected_records),
    "duplicate_records": int(duplicate_count),
    "silver_output_records": int(silver_records),
    "suspicious_records": int(suspicious_records),
    "silver_output_path": str(silver_output_path),
    "rejected_output_path": str(rejected_output_path),
    "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}]

phase04_audit_df = spark.createDataFrame(phase04_audit_data)

phase04_audit_df.show(truncate=False)

audit_output_path = AUDIT_PATH / f"phase04_transformation_audit_{PROCESS_ID}"

(
    phase04_audit_df
    .write
    .mode("overwrite")
    .parquet(spark_path(audit_output_path))
)

print("Auditoría de Fase 04 guardada en:")
print(audit_output_path)

+--------------------+-----------------+----------------------------+---------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------+----------------+------------------------------------------------------------------------------------------------------------------------------+---------------------+------------------+
|bronze_input_records|duplicate_records|phase                       |process_id     |processed_at       |rejected_output_path                                                                                                                     |rejected_records|silver_output_path                                                                                                            |silver_output_records|suspicious_records|
+--------------------+-----------------+----------------------------+---------------+-------------------+-------------------------------------

## 21. Validaciones finales de Fase 04

Se comprueba que el dataset transformado incluya los campos derivados mínimos y que los registros se hayan distribuido por servicio, año y mes.


In [24]:
# Validar campos derivados mínimos.

derived_required_columns = [
    "trip_duration_minutes",
    "average_speed_mph",
    "fare_per_mile",
    "tip_percentage",
    "is_airport_trip",
    "is_suspicious_trip",
    "processing_date"
]

missing_derived_columns = [col for col in derived_required_columns if col not in trips_silver.columns]

if missing_derived_columns:
    print("Faltan columnas derivadas:", missing_derived_columns)
else:
    print("Todas las columnas derivadas mínimas están presentes.")

print("Distribución por servicio, año y mes:")
trips_silver.groupBy("service_type", "year", "month").count().orderBy("service_type", "year", "month").show(50, truncate=False)

print("Resumen de indicadores:")
trips_silver.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("is_airport_trip").cast("int")).alias("airport_trips"),
    F.sum(F.col("is_suspicious_trip").cast("int")).alias("suspicious_trips"),
    F.avg("trip_duration_minutes").alias("avg_duration"),
    F.avg("average_speed_mph").alias("avg_speed"),
    F.avg("fare_per_mile").alias("avg_fare_per_mile"),
    F.avg("tip_percentage").alias("avg_tip_percentage")
).show(truncate=False)

Todas las columnas derivadas mínimas están presentes.
Distribución por servicio, año y mes:
+------------+----+-----+--------+
|service_type|year|month|count   |
+------------+----+-----+--------+
|fhvhv       |2023|1    |18452962|
|green       |2023|1    |64255   |
|green       |2023|2    |61381   |
|yellow      |2020|1    |6281130 |
|yellow      |2021|1    |1331340 |
|yellow      |2022|1    |2411789 |
|yellow      |2022|2    |2920554 |
|yellow      |2023|1    |2991419 |
|yellow      |2023|2    |2843569 |
|yellow      |2023|3    |3319914 |
|yellow      |2023|4    |3212847 |
+------------+----+-----+--------+

Resumen de indicadores:
+-------------+-------------+----------------+------------------+------------------+-----------------+------------------+
|total_records|airport_trips|suspicious_trips|avg_duration      |avg_speed         |avg_fare_per_mile|avg_tip_percentage|
+-------------+-------------+----------------+------------------+------------------+-----------------+------------

## Conclusión de la Fase 04

En esta fase se transformó el dataset reconstruido desde Bronze para generar una salida limpia y analíticamente útil en Silver.

Se normalizaron nombres de columnas, se convirtieron tipos de datos, se calcularon campos derivados, se generó `trip_id`, se marcaron viajes sospechosos, se separaron registros inválidos y se eliminaron duplicados técnicos.

Además, se guardaron los registros rechazados en cuarentena y se generó una auditoría técnica de la fase. Con esto, el dataset queda preparado para la siguiente etapa: **validación de calidad de datos y generación de métricas**.


# Fase 05: Validación de calidad de datos

## Objetivo

Esta fase mide y demuestra la calidad del dataset transformado antes de cargarlo a la base de datos.

La validación se realiza con Apache Spark/PySpark y genera dos salidas obligatorias:

1. `quality_rejected_records`: tabla de registros rechazados con la causa técnica y de negocio.
2. `quality_metrics_summary`: resumen de métricas de calidad por servicio, año y mes.

## Reglas evaluadas

Se validan las reglas mínimas solicitadas en el caso de estudio:

- Nulidad de campos críticos.
- Rangos válidos de fechas.
- Rangos válidos de montos.
- Rangos válidos de distancia.
- Rangos válidos de duración.
- Duplicados técnicos.
- Integridad de particiones.
- Consistencia entre año/mes de partición y fecha real del viaje.
- Porcentaje de registros rechazados.
- Distribución de registros por servicio.
- Detección de outliers.

In [25]:
# Preparar Python, Hadoop y Spark para trabajar en Windows.
# Esta celda evita errores donde Spark intenta buscar python3 en Windows.

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

print("Python usado por PySpark:")
print(sys.executable)
print("Existe winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("Existe hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

Python usado por PySpark:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
Existe winutils: False
Existe hadoop.dll: False


In [26]:
# Importar librerías principales.

from pathlib import Path
from datetime import datetime
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

In [27]:
# Crear o reutilizar la sesión Spark.

spark = (
    SparkSession.builder
    .appName("Fase_05_Validacion_Calidad_Datos")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.io.native.lib.available", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente")
print("Versión Spark:", spark.version)

Spark iniciado correctamente
Versión Spark: 4.1.2


In [28]:
# Definir rutas principales del proyecto.
# Si el notebook se ejecuta desde la carpeta notebooks, sube un nivel.

BASE_PATH = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

DATA_PATH = BASE_PATH / "data"
SILVER_PATH = DATA_PATH / "silver"
QUARANTINE_PATH = DATA_PATH / "quarantine"
AUDIT_PATH = DATA_PATH / "audit"

PROCESS_ID = "fase5_" + datetime.now().strftime("%Y%m%d_%H%M%S")

SILVER_PATH.mkdir(parents=True, exist_ok=True)
QUARANTINE_PATH.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("PROCESS_ID:", PROCESS_ID)

BASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
SILVER_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver
QUARANTINE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
PROCESS_ID: fase5_20260617_181600


In [29]:
# Función auxiliar para convertir rutas Windows a formato legible por Spark.

def spark_path(path):
    return str(Path(path).resolve()).replace("\\", "/")

print("Ejemplo ruta Spark:")
print(spark_path(SILVER_PATH))

Ejemplo ruta Spark:
C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/silver


## 1. Carga del dataset transformado desde Silver

La Fase 05 parte del resultado generado por la Fase 04. Por eso se busca automáticamente una carpeta válida en `data/silver` que contenga archivos `.parquet` reales.

Se ignoran carpetas temporales, archivos ocultos, salidas de cuarentena y salidas de auditoría para evitar leer resultados incorrectos.

In [30]:
# Buscar la carpeta Silver válida más reciente.
# Se descartan carpetas incompletas, temporales o relacionadas con métricas/rechazados.

EXCLUDED_KEYWORDS = [
    "_temporary",
    "quality_rejected_records",
    "quality_metrics_summary",
    "rejected",
    "quarantine",
    "metrics",
    "audit"
]

silver_candidates = []

if SILVER_PATH.exists():
    for path in SILVER_PATH.iterdir():
        if path.is_dir():
            path_text = str(path).lower()
            if any(keyword in path_text for keyword in EXCLUDED_KEYWORDS):
                continue

            parquet_files = [
                file for file in path.rglob("*.parquet")
                if "_temporary" not in str(file).lower()
                and not file.name.startswith(".")
                and file.name.endswith(".parquet")
            ]

            if len(parquet_files) > 0:
                latest_file_time = max(file.stat().st_mtime for file in parquet_files)
                silver_candidates.append({
                    "path": path,
                    "modified_time": latest_file_time,
                    "parquet_count": len(parquet_files)
                })

if len(silver_candidates) == 0:
    raise FileNotFoundError(
        "No se encontró una carpeta Silver válida con archivos .parquet reales. "
        "Primero ejecuta correctamente la Fase 04 y guarda el dataset transformado en data/silver."
    )

latest_silver_info = max(silver_candidates, key=lambda item: item["modified_time"])
latest_silver_path = latest_silver_info["path"]

print("Dataset Silver válido seleccionado:")
print(latest_silver_path)
print("Archivos parquet encontrados:", latest_silver_info["parquet_count"])

Dataset Silver válido seleccionado:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver\trips_transformed_20260617_181025
Archivos parquet encontrados: 44


In [31]:
# Leer dataset Silver válido.
# Se lee la carpeta raíz del Silver particionado para que Spark recupere columnas de partición:
# service_type, year y month.

silver_candidates = []

if SILVER_PATH.exists():
    for path in SILVER_PATH.glob("trips_transformed_*"):
        if path.is_dir():
            parquet_files = [
                file for file in path.rglob("*.parquet")
                if "_temporary" not in str(file)
                and not file.name.startswith(".")
            ]

            if len(parquet_files) > 0:
                silver_candidates.append({
                    "path": path,
                    "modified_time": path.stat().st_mtime,
                    "parquet_count": len(parquet_files)
                })

if len(silver_candidates) == 0:
    raise FileNotFoundError("No se encontró una carpeta Silver válida.")

latest_silver_info = max(silver_candidates, key=lambda item: item["modified_time"])
latest_silver_path = latest_silver_info["path"]

print("Silver seleccionado:")
print(latest_silver_path)
print("Archivos parquet encontrados:", latest_silver_info["parquet_count"])

silver_trips = spark.read.parquet(spark_path(latest_silver_path))

print("Total Silver:", silver_trips.count())
silver_trips.groupBy("service_type", "year", "month").count().show(50, truncate=False)

Silver seleccionado:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver\trips_transformed_20260617_181025
Archivos parquet encontrados: 44
Total Silver: 43891160
+------------+----+-----+--------+
|service_type|year|month|count   |
+------------+----+-----+--------+
|fhvhv       |2023|1    |18452962|
|yellow      |2020|1    |6281130 |
|yellow      |2023|3    |3319914 |
|yellow      |2023|4    |3212847 |
|yellow      |2023|1    |2991419 |
|yellow      |2022|2    |2920554 |
|yellow      |2023|2    |2843569 |
|yellow      |2022|1    |2411789 |
|yellow      |2021|1    |1331340 |
|green       |2023|2    |61381   |
|green       |2023|1    |64255   |
+------------+----+-----+--------+



## 2. Estandarización preventiva del esquema

Antes de aplicar reglas de calidad se verifica que existan las columnas necesarias. Si una columna requerida no existe, se crea con valor nulo controlado para que las validaciones sean auditables y el pipeline no falle.

In [32]:
# Columnas mínimas requeridas para la validación de calidad.

required_columns = {
    "process_id": "string",
    "trip_id": "string",
    "service_type": "string",
    "source_file": "string",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "trip_distance": "double",
    "fare_amount": "double",
    "total_amount": "double",
    "tip_amount": "double",
    "trip_duration_minutes": "double",
    "average_speed_mph": "double",
    "fare_per_mile": "double",
    "tip_percentage": "double",
    "is_suspicious_trip": "boolean",
    "year": "int",
    "month": "int",
    "quality_status": "string",
    "processing_date": "date"
}

quality_input = silver_trips

for column_name, data_type in required_columns.items():
    if column_name not in quality_input.columns:
        quality_input = quality_input.withColumn(column_name, F.lit(None).cast(data_type))
    else:
        quality_input = quality_input.withColumn(column_name, F.col(column_name).cast(data_type))

# Si la Fase 04 no heredó process_id, se asigna el process_id de esta ejecución.
quality_input = quality_input.withColumn(
    "process_id",
    F.when(F.col("process_id").isNull(), F.lit(PROCESS_ID)).otherwise(F.col("process_id"))
)

print("Esquema estandarizado para validación de calidad:")
quality_input.printSchema()
quality_input.show(5, truncate=False)

Esquema estandarizado para validación de calidad:
root
 |-- trip_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- average_speed_mph: double (nullable = true)
 |-- fare_per_mile: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double

## 3. Definición de reglas de calidad

Cada regla genera registros rechazados con la estructura obligatoria:

- `process_id`
- `trip_id`
- `service_type`
- `source_file`
- `rejection_stage`
- `rejection_rule`
- `rejection_column`
- `original_value`
- `technical_reason`
- `business_reason`
- `rejected_at`

Un mismo viaje puede incumplir más de una regla, por eso la tabla de rechazados conserva una fila por cada regla incumplida.

In [33]:
# Esquema obligatorio para quality_rejected_records.

quality_rejected_schema = StructType([
    StructField("process_id", StringType(), True),
    StructField("trip_id", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("rejection_stage", StringType(), True),
    StructField("rejection_rule", StringType(), True),
    StructField("rejection_column", StringType(), True),
    StructField("original_value", StringType(), True),
    StructField("technical_reason", StringType(), True),
    StructField("business_reason", StringType(), True),
    StructField("rejected_at", TimestampType(), True)
])


def build_rejection_df(df, condition, rejection_rule, rejection_column, original_value_expr, technical_reason, business_reason):
    """
    Construye un DataFrame de registros rechazados para una regla específica.
    """
    return (
        df.filter(condition)
        .select(
            F.col("process_id").cast("string").alias("process_id"),
            F.col("trip_id").cast("string").alias("trip_id"),
            F.col("service_type").cast("string").alias("service_type"),
            F.col("source_file").cast("string").alias("source_file"),
            F.lit("FASE_05_VALIDACION_CALIDAD").alias("rejection_stage"),
            F.lit(rejection_rule).alias("rejection_rule"),
            F.lit(rejection_column).alias("rejection_column"),
            original_value_expr.cast("string").alias("original_value"),
            F.lit(technical_reason).alias("technical_reason"),
            F.lit(business_reason).alias("business_reason"),
            F.current_timestamp().alias("rejected_at")
        )
    )

In [34]:
# Regla 1: nulidad de campos críticos.

critical_columns = [
    "trip_id",
    "service_type",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "total_amount",
    "source_file"
]

# Crear columnas faltantes usadas como críticas si no existen todavía.
for column_name in critical_columns:
    if column_name not in quality_input.columns:
        quality_input = quality_input.withColumn(column_name, F.lit(None).cast("string"))

null_critical_condition = reduce(
    lambda a, b: a | b,
    [F.col(column).isNull() for column in critical_columns]
)

null_fields_expr = F.concat_ws(
    ",",
    *[
        F.when(F.col(column).isNull(), F.lit(column)).otherwise(F.lit(None))
        for column in critical_columns
    ]
)

rejected_nulls = build_rejection_df(
    quality_input,
    null_critical_condition,
    "NULL_CRITICAL_FIELDS",
    "critical_columns",
    null_fields_expr,
    "Existen valores nulos en campos críticos requeridos para análisis.",
    "El viaje no puede considerarse confiable si faltan identificadores, fechas, ubicación, monto o archivo fuente."
)

print("Registros con nulos críticos:", rejected_nulls.count())
rejected_nulls.show(10, truncate=False)

Registros con nulos críticos: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [35]:
# Regla 2: rangos válidos de fechas.
# Se consideran inválidas fechas nulas, fechas invertidas, fechas futuras o fechas fuera de un rango razonable.

current_timestamp = F.current_timestamp()

invalid_date_condition = (
    F.col("pickup_datetime").isNull() |
    F.col("dropoff_datetime").isNull() |
    (F.col("pickup_datetime") > F.col("dropoff_datetime")) |
    (F.col("pickup_datetime") > current_timestamp) |
    (F.year("pickup_datetime") < F.lit(2000)) |
    (F.year("pickup_datetime") > F.year(current_timestamp))
)

rejected_dates = build_rejection_df(
    quality_input,
    invalid_date_condition,
    "INVALID_DATE_RANGE",
    "pickup_datetime/dropoff_datetime",
    F.concat_ws(" | ", F.col("pickup_datetime"), F.col("dropoff_datetime")),
    "La fecha de recogida o llegada es nula, futura, invertida o fuera del rango esperado.",
    "Un viaje con fechas inválidas altera cálculos de duración, demanda y análisis temporal."
)

print("Registros con fechas inválidas:", rejected_dates.count())
rejected_dates.show(10, truncate=False)

Registros con fechas inválidas: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [36]:
# Regla 3: rangos válidos de montos.
# Se rechazan montos negativos o total_amount menor o igual a cero.

invalid_amount_condition = (
    F.col("total_amount").isNull() |
    (F.col("total_amount") <= 0) |
    (F.col("fare_amount") < 0) |
    (F.col("tip_amount") < 0)
)

rejected_amounts = build_rejection_df(
    quality_input,
    invalid_amount_condition,
    "INVALID_AMOUNT_RANGE",
    "total_amount/fare_amount/tip_amount",
    F.concat_ws(" | ", F.col("total_amount"), F.col("fare_amount"), F.col("tip_amount")),
    "Existen montos nulos, negativos o total pagado menor o igual a cero.",
    "Los montos inválidos afectan ingresos, propinas y cualquier indicador financiero."
)

print("Registros con montos inválidos:", rejected_amounts.count())
rejected_amounts.show(10, truncate=False)

Registros con montos inválidos: 15
+---------------------+----------------------------------------------------------------+------------+-------------------------------+--------------------------+--------------------+-----------------------------------+---------------------+--------------------------------------------------------------------+---------------------------------------------------------------------------------+--------------------------+
|process_id           |trip_id                                                         |service_type|source_file                    |rejection_stage           |rejection_rule      |rejection_column                   |original_value       |technical_reason                                                    |business_reason                                                                  |rejected_at               |
+---------------------+----------------------------------------------------------------+------------+----------------------------

In [37]:
# Regla 4: rangos válidos de distancia.
# Se rechazan distancias nulas, cero, negativas o excesivamente altas.

invalid_distance_condition = (
    F.col("trip_distance").isNull() |
    (F.col("trip_distance") <= 0) |
    (F.col("trip_distance") > 300)
)

rejected_distance = build_rejection_df(
    quality_input,
    invalid_distance_condition,
    "INVALID_DISTANCE_RANGE",
    "trip_distance",
    F.col("trip_distance"),
    "La distancia del viaje es nula, menor o igual a cero, o supera un umbral operativo razonable.",
    "Una distancia inválida afecta velocidad, tarifa por milla y análisis de movilidad."
)

print("Registros con distancia inválida:", rejected_distance.count())
rejected_distance.show(10, truncate=False)

Registros con distancia inválida: 2
+---------------------+----------------------------------------------------------------+------------+------------------------------+--------------------------+----------------------+----------------+--------------+---------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------+--------------------------+
|process_id           |trip_id                                                         |service_type|source_file                   |rejection_stage           |rejection_rule        |rejection_column|original_value|technical_reason                                                                             |business_reason                                                                   |rejected_at               |
+---------------------+----------------------------------------------------------------+------------+-------------------------

In [38]:
# Regla 5: rangos válidos de duración.
# Se rechazan duraciones nulas, negativas, cero o mayores a 480 minutos.

invalid_duration_condition = (
    F.col("trip_duration_minutes").isNull() |
    (F.col("trip_duration_minutes") <= 0) |
    (F.col("trip_duration_minutes") > 480)
)

rejected_duration = build_rejection_df(
    quality_input,
    invalid_duration_condition,
    "INVALID_DURATION_RANGE",
    "trip_duration_minutes",
    F.col("trip_duration_minutes"),
    "La duración del viaje es nula, negativa, cero o mayor a 480 minutos.",
    "Una duración inválida distorsiona tiempos promedio, velocidad y eficiencia operacional."
)

print("Registros con duración inválida:", rejected_duration.count())
rejected_duration.show(10, truncate=False)

Registros con duración inválida: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [39]:
# Regla 6: duplicados técnicos.
# Se identifican duplicados por trip_id y también por una llave técnica de viaje.

technical_duplicate_columns = [
    "service_type",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_location_id",
    "dropoff_location_id",
    "trip_distance",
    "total_amount",
    "source_file"
]

window_trip_id = Window.partitionBy("trip_id")
window_technical = Window.partitionBy(*technical_duplicate_columns)

quality_with_duplicates = (
    quality_input
    .withColumn("trip_id_duplicate_count", F.count("*").over(window_trip_id))
    .withColumn("technical_duplicate_count", F.count("*").over(window_technical))
)

duplicate_condition = (
    (F.col("trip_id_duplicate_count") > 1) |
    (F.col("technical_duplicate_count") > 1)
)

rejected_duplicates = build_rejection_df(
    quality_with_duplicates,
    duplicate_condition,
    "TECHNICAL_DUPLICATE_RECORD",
    "trip_id/technical_key",
    F.concat_ws(" | ", F.col("trip_id"), F.col("trip_id_duplicate_count"), F.col("technical_duplicate_count")),
    "El registro aparece duplicado por trip_id o por llave técnica del viaje.",
    "Los duplicados inflan viajes, ingresos y métricas operativas."
)

print("Registros duplicados:", rejected_duplicates.count())
rejected_duplicates.show(10, truncate=False)

Registros duplicados: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [40]:
# Regla 7: integridad de particiones.
# Se valida que year y month existan y estén en rangos válidos.

invalid_partition_condition = (
    F.col("year").isNull() |
    F.col("month").isNull() |
    (F.col("year") < 2000) |
    (F.col("year") > F.year(F.current_timestamp())) |
    (F.col("month") < 1) |
    (F.col("month") > 12)
)

rejected_partition_integrity = build_rejection_df(
    quality_input,
    invalid_partition_condition,
    "INVALID_PARTITION_VALUES",
    "year/month",
    F.concat_ws(" | ", F.col("year"), F.col("month")),
    "Las columnas de partición year/month son nulas o están fuera de rango.",
    "La partición inválida impide organizar correctamente los datos por periodo."
)

print("Registros con particiones inválidas:", rejected_partition_integrity.count())
rejected_partition_integrity.show(10, truncate=False)

Registros con particiones inválidas: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [41]:
# Regla 8: consistencia entre año/mes de partición y fecha real del viaje.

partition_date_mismatch_condition = (
    F.col("pickup_datetime").isNotNull() &
    F.col("year").isNotNull() &
    F.col("month").isNotNull() &
    (
        (F.col("year") != F.year("pickup_datetime")) |
        (F.col("month") != F.month("pickup_datetime"))
    )
)

rejected_partition_consistency = build_rejection_df(
    quality_input,
    partition_date_mismatch_condition,
    "PARTITION_DATE_MISMATCH",
    "year/month/pickup_datetime",
    F.concat_ws(" | ", F.col("year"), F.col("month"), F.col("pickup_datetime")),
    "La partición year/month no coincide con la fecha real de pickup_datetime.",
    "La inconsistencia de partición afecta consultas por periodo y reportes mensuales."
)

print("Registros con inconsistencia año/mes vs fecha:", rejected_partition_consistency.count())
rejected_partition_consistency.show(10, truncate=False)

Registros con inconsistencia año/mes vs fecha: 614
+---------------------+----------------------------------------------------------------+------------+-------------------------------+--------------------------+-----------------------+--------------------------+------------------------------+-------------------------------------------------------------------------+---------------------------------------------------------------------------------+--------------------------+
|process_id           |trip_id                                                         |service_type|source_file                    |rejection_stage           |rejection_rule         |rejection_column          |original_value                |technical_reason                                                         |business_reason                                                                  |rejected_at               |
+---------------------+----------------------------------------------------------------+---------

In [42]:
# Regla 9: viajes sospechosos según reglas avanzadas.
# Se reutiliza is_suspicious_trip generado en Fase 04 y también se refuerzan condiciones críticas.

suspicious_condition = (
    (F.col("is_suspicious_trip") == True) |
    (F.col("trip_distance") <= 0) |
    (F.col("total_amount") <= 0) |
    (F.col("fare_amount") < 0) |
    (F.col("trip_duration_minutes") <= 0) |
    (F.col("trip_duration_minutes") > 480) |
    (F.col("average_speed_mph") > 100) |
    (F.col("tip_percentage") > 100) |
    (F.col("pickup_datetime") > F.col("dropoff_datetime")) |
    (F.col("pickup_datetime") > F.current_timestamp())
)

rejected_suspicious = build_rejection_df(
    quality_input,
    suspicious_condition,
    "SUSPICIOUS_TRIP_RULES",
    "is_suspicious_trip",
    F.concat_ws(
        " | ",
        F.col("trip_distance"),
        F.col("total_amount"),
        F.col("fare_amount"),
        F.col("trip_duration_minutes"),
        F.col("average_speed_mph"),
        F.col("tip_percentage")
    ),
    "El viaje cumple una o más reglas avanzadas de sospecha.",
    "Los viajes sospechosos requieren revisión antes de ser usados en análisis de negocio."
)

print("Registros sospechosos:", rejected_suspicious.count())
rejected_suspicious.show(10, truncate=False)

Registros sospechosos: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



In [43]:
# Regla 10: detección de outliers estadísticos.
# Se calcula el percentil 99 para distancia, total pagado y duración.

numeric_outlier_columns = ["trip_distance", "total_amount", "trip_duration_minutes"]
outlier_thresholds = {}

for column in numeric_outlier_columns:
    try:
        quantile_values = quality_input.select(column).na.drop().approxQuantile(column, [0.99], 0.01)
        outlier_thresholds[column] = quantile_values[0] if len(quantile_values) > 0 else None
    except Exception:
        outlier_thresholds[column] = None

print("Umbrales P99 calculados para outliers:")
print(outlier_thresholds)

outlier_condition = F.lit(False)
for column, threshold in outlier_thresholds.items():
    if threshold is not None and threshold > 0:
        outlier_condition = outlier_condition | (F.col(column) > F.lit(threshold))

rejected_outliers = build_rejection_df(
    quality_input,
    outlier_condition,
    "STATISTICAL_OUTLIER_P99",
    "trip_distance/total_amount/trip_duration_minutes",
    F.concat_ws(" | ", F.col("trip_distance"), F.col("total_amount"), F.col("trip_duration_minutes")),
    "El registro supera el percentil 99 en una o más variables numéricas relevantes.",
    "Los outliers pueden distorsionar promedios, ingresos y análisis operativo."
)

print("Registros outliers:", rejected_outliers.count())
rejected_outliers.show(10, truncate=False)

Umbrales P99 calculados para outliers:
{'trip_distance': 351.17, 'total_amount': 401095.62, 'trip_duration_minutes': 480.0}
Registros outliers: 0
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
|process_id|trip_id|service_type|source_file|rejection_stage|rejection_rule|rejection_column|original_value|technical_reason|business_reason|rejected_at|
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+
+----------+-------+------------+-----------+---------------+--------------+----------------+--------------+----------------+---------------+-----------+



## 4. Construcción de `quality_rejected_records`

En esta sección se consolidan todas las reglas de calidad en una única tabla obligatoria de registros rechazados.

In [44]:
# Construcción segura de quality_rejected_records

from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType

# Evita que Spark se caiga por archivos problemáticos al leer parquet
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

required_rejected_columns = [
    "process_id",
    "trip_id",
    "service_type",
    "source_file",
    "rejection_stage",
    "rejection_rule",
    "rejection_column",
    "original_value",
    "technical_reason",
    "business_reason",
    "rejected_at"
]

rejection_dfs = [
    rejected_nulls,
    rejected_dates,
    rejected_amounts,
    rejected_distance,
    rejected_duration,
    rejected_duplicates,
    rejected_partition_integrity,
    rejected_partition_consistency,
    rejected_suspicious,
    rejected_outliers
]

# Función para asegurar que todos los DataFrames tengan las mismas columnas
def normalize_rejected_df(df):
    for col_name in required_rejected_columns:
        if col_name not in df.columns:
            if col_name == "rejected_at":
                df = df.withColumn(col_name, F.current_timestamp())
            else:
                df = df.withColumn(col_name, F.lit(None).cast(StringType()))

    return df.select(required_rejected_columns)

# Normalizar todos los DataFrames
rejection_dfs_normalized = [
    normalize_rejected_df(df)
    for df in rejection_dfs
]

# Unir todos los rechazados
quality_rejected_records = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    rejection_dfs_normalized
)

# Quitar duplicados de la misma regla para el mismo viaje
quality_rejected_records = (
    quality_rejected_records
    .dropDuplicates(["trip_id", "rejection_rule", "rejection_column"])
)

# Guardar en memoria/disco para cortar linaje pesado
quality_rejected_records = quality_rejected_records.persist()

print("Tabla quality_rejected_records construida correctamente.")

# Mostrar una muestra pequeña sin forzar demasiado Spark
quality_rejected_records.limit(20).show(truncate=False)

quality_rejected_records.printSchema()

Tabla quality_rejected_records construida correctamente.
+---------------------+----------------------------------------------------------------+------------+-------------------------------+--------------------------+-----------------------+-----------------------------------+------------------------------+-------------------------------------------------------------------------+---------------------------------------------------------------------------------+-------------------------+
|process_id           |trip_id                                                         |service_type|source_file                    |rejection_stage           |rejection_rule         |rejection_column                   |original_value                |technical_reason                                                         |business_reason                                                                  |rejected_at              |
+---------------------+----------------------------------------------------

In [45]:
# Guardar tabla obligatoria quality_rejected_records en quarantine.

quality_rejected_output_path = QUARANTINE_PATH / f"quality_rejected_records_{PROCESS_ID}"

(
    quality_rejected_records
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(spark_path(quality_rejected_output_path))
)

print("quality_rejected_records guardado en:")
print(quality_rejected_output_path)

quality_rejected_records guardado en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\quality_rejected_records_fase5_20260617_181600


## 5. Separación de registros válidos e inválidos

Para calcular métricas de calidad y preparar datos confiables para capas posteriores, se separan los viajes con al menos una regla incumplida de los viajes que no presentan rechazos.

In [46]:
# Identificar viajes rechazados de manera única.

rejected_trip_ids = (
    quality_rejected_records
    .select("trip_id")
    .where(F.col("trip_id").isNotNull())
    .dropDuplicates()
)

valid_trips = quality_input.join(rejected_trip_ids, on="trip_id", how="left_anti")
invalid_trips = quality_input.join(rejected_trip_ids, on="trip_id", how="inner")

valid_trips = valid_trips.withColumn("quality_status", F.lit("VALID"))
invalid_trips = invalid_trips.withColumn("quality_status", F.lit("REJECTED"))

print("Registros válidos:", valid_trips.count())
print("Registros inválidos únicos:", invalid_trips.count())

valid_trips.show(5, truncate=False)
invalid_trips.show(5, truncate=False)

Registros válidos: 43890529
Registros inválidos únicos: 631
+----------------------------------------------------------------+---------+-------------------+-------------------+---------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------+-----------+------------+-------+----------+------------+------------------+--------------+--------------------+-----------+---------------+------------------+------------------------------+--------------------------+---------------+--------------+------------+----+-----+---------------------+
|trip_id                                                         |vendor_id|pickup_datetime    |dropoff_datetime   |trip_duration_minutes|passenger_count|trip_distance|average_speed_mph |fare_per_mile     |pickup_location_id|dropoff_location_id|payment_type|fare_amount|extra_amount|mta_tax|tip_amount|tolls_amount|total_amount      |tip_percentage|congestion_surcharge|airport_fee|is_ai

In [47]:
# Guardar dataset validado en Silver para consumo de la Fase 06.

validated_output_path = SILVER_PATH / f"validated_trips_{PROCESS_ID}"

(
    valid_trips
    .repartition("service_type", "year", "month")
    .write
    .mode("overwrite")
    .partitionBy("service_type", "year", "month")
    .parquet(spark_path(validated_output_path))
)

print("Dataset validado guardado en Silver:")
print(validated_output_path)

Dataset validado guardado en Silver:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver\validated_trips_fase5_20260617_181600


## 6. Tabla obligatoria `quality_metrics_summary`

La tabla de métricas resume la calidad por `service_type`, `year` y `month`. Incluye registros totales, válidos, rechazados, duplicados, nulos críticos, sospechosos y porcentaje de calidad.

In [48]:
# Preparar conteos base por servicio, año y mes.

group_cols = ["service_type", "year", "month"]

base_counts = (
    quality_input
    .groupBy(*group_cols)
    .agg(F.count("*").alias("total_records"))
)

valid_counts = (
    valid_trips
    .groupBy(*group_cols)
    .agg(F.count("*").alias("valid_records"))
)

rejected_counts = (
    invalid_trips
    .groupBy(*group_cols)
    .agg(F.count("*").alias("rejected_records"))
)

duplicate_counts = (
    rejected_duplicates
    .select("trip_id")
    .dropDuplicates()
    .join(quality_input, on="trip_id", how="inner")
    .groupBy(*group_cols)
    .agg(F.countDistinct("trip_id").alias("duplicate_records"))
)

null_critical_counts = (
    rejected_nulls
    .select("trip_id")
    .dropDuplicates()
    .join(quality_input, on="trip_id", how="inner")
    .groupBy(*group_cols)
    .agg(F.countDistinct("trip_id").alias("null_critical_records"))
)

suspicious_counts = (
    quality_input
    .where(F.col("is_suspicious_trip") == True)
    .groupBy(*group_cols)
    .agg(F.count("*").alias("suspicious_records"))
)

quality_metrics_summary = (
    base_counts
    .join(valid_counts, on=group_cols, how="left")
    .join(rejected_counts, on=group_cols, how="left")
    .join(duplicate_counts, on=group_cols, how="left")
    .join(null_critical_counts, on=group_cols, how="left")
    .join(suspicious_counts, on=group_cols, how="left")
    .na.fill({
        "valid_records": 0,
        "rejected_records": 0,
        "duplicate_records": 0,
        "null_critical_records": 0,
        "suspicious_records": 0
    })
    .withColumn("process_id", F.lit(PROCESS_ID))
    .withColumn(
        "quality_percentage",
        F.when(F.col("total_records") > 0,
               F.round((F.col("valid_records") / F.col("total_records")) * 100, 2))
        .otherwise(F.lit(0.0))
    )
    .withColumn("processed_at", F.current_timestamp())
    .select(
        "process_id",
        "service_type",
        "year",
        "month",
        "total_records",
        "valid_records",
        "rejected_records",
        "duplicate_records",
        "null_critical_records",
        "suspicious_records",
        "quality_percentage",
        "processed_at"
    )
    .orderBy("year", "month", "service_type")
)

print("Tabla quality_metrics_summary generada:")
quality_metrics_summary.show(50, truncate=False)
quality_metrics_summary.printSchema()

Tabla quality_metrics_summary generada:
+---------------------+------------+----+-----+-------------+-------------+----------------+-----------------+---------------------+------------------+------------------+-------------------------+
|process_id           |service_type|year|month|total_records|valid_records|rejected_records|duplicate_records|null_critical_records|suspicious_records|quality_percentage|processed_at             |
+---------------------+------------+----+-----+-------------+-------------+----------------+-----------------+---------------------+------------------+------------------+-------------------------+
|fase5_20260617_181600|yellow      |2020|1    |6281130      |6280948      |182             |0                |0                    |0                 |100.0             |2026-06-17 18:24:57.96254|
|fase5_20260617_181600|yellow      |2021|1    |1331340      |1331320      |20              |0                |0                    |0                 |100.0             |20

In [49]:
# Guardar tabla obligatoria quality_metrics_summary en audit.

quality_metrics_output_path = AUDIT_PATH / f"quality_metrics_summary_{PROCESS_ID}"

(
    quality_metrics_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(spark_path(quality_metrics_output_path))
)

print("quality_metrics_summary guardado en:")
print(quality_metrics_output_path)

quality_metrics_summary guardado en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\quality_metrics_summary_fase5_20260617_181600


## 7. Porcentaje de rechazados y distribución por servicio

Estas salidas sirven como evidencia visual de la calidad del dataset antes de pasar a la capa Gold y a la base de datos.

In [50]:
# Porcentaje general de registros rechazados.

total_records = quality_input.count()
unique_rejected_records = rejected_trip_ids.count()
valid_records = valid_trips.count()

rejected_percentage = round((unique_rejected_records / total_records) * 100, 2) if total_records > 0 else 0
quality_percentage = round((valid_records / total_records) * 100, 2) if total_records > 0 else 0

print("Resumen general de calidad")
print("Total de registros:", total_records)
print("Registros válidos:", valid_records)
print("Registros rechazados únicos:", unique_rejected_records)
print("Porcentaje de rechazo:", rejected_percentage, "%")
print("Porcentaje de calidad:", quality_percentage, "%")

Resumen general de calidad
Total de registros: 43891160
Registros válidos: 43890529
Registros rechazados únicos: 631
Porcentaje de rechazo: 0.0 %
Porcentaje de calidad: 100.0 %


In [51]:
# Distribución de registros por servicio.

distribution_by_service = (
    quality_input
    .groupBy("service_type")
    .agg(F.count("*").alias("total_records"))
    .orderBy(F.desc("total_records"))
)

print("Distribución de registros por servicio:")
distribution_by_service.show(truncate=False)

Distribución de registros por servicio:
+------------+-------------+
|service_type|total_records|
+------------+-------------+
|yellow      |25312562     |
|fhvhv       |18452962     |
|green       |125636       |
+------------+-------------+



In [52]:
# Resumen de rechazos por regla.

rejection_rule_summary = (
    quality_rejected_records
    .groupBy("rejection_rule")
    .agg(F.count("*").alias("total_rejections"))
    .orderBy(F.desc("total_rejections"))
)

print("Resumen de rechazos por regla:")
rejection_rule_summary.show(50, truncate=False)

Resumen de rechazos por regla:
+-----------------------+----------------+
|rejection_rule         |total_rejections|
+-----------------------+----------------+
|PARTITION_DATE_MISMATCH|614             |
|INVALID_AMOUNT_RANGE   |15              |
|INVALID_DISTANCE_RANGE |2               |
+-----------------------+----------------+



## 8. Validación final de cumplimiento de la Fase 05

Esta celda verifica que existan las columnas obligatorias en las dos tablas principales de la fase.

In [53]:
# Validación final de columnas obligatorias.

required_rejected_columns = [
    "process_id",
    "trip_id",
    "service_type",
    "source_file",
    "rejection_stage",
    "rejection_rule",
    "rejection_column",
    "original_value",
    "technical_reason",
    "business_reason",
    "rejected_at"
]

required_metrics_columns = [
    "process_id",
    "service_type",
    "year",
    "month",
    "total_records",
    "valid_records",
    "rejected_records",
    "duplicate_records",
    "null_critical_records",
    "suspicious_records",
    "quality_percentage",
    "processed_at"
]

missing_rejected_columns = [col for col in required_rejected_columns if col not in quality_rejected_records.columns]
missing_metrics_columns = [col for col in required_metrics_columns if col not in quality_metrics_summary.columns]

print("Columnas faltantes en quality_rejected_records:", missing_rejected_columns)
print("Columnas faltantes en quality_metrics_summary:", missing_metrics_columns)

if len(missing_rejected_columns) == 0 and len(missing_metrics_columns) == 0:
    print("Validación final correcta: las tablas obligatorias cumplen el esquema solicitado.")
else:
    raise Exception("Existen columnas obligatorias faltantes en las tablas de calidad.")

Columnas faltantes en quality_rejected_records: []
Columnas faltantes en quality_metrics_summary: []
Validación final correcta: las tablas obligatorias cumplen el esquema solicitado.


## Conclusión de la Fase 05

En esta fase se validó la calidad del dataset transformado antes de cargarlo a una base de datos. Se aplicaron reglas sobre campos críticos, fechas, montos, distancia, duración, duplicados, particiones, consistencia temporal y outliers.

Los registros que incumplen reglas fueron enviados a `quality_rejected_records`, conservando la causa técnica y de negocio del rechazo. También se generó `quality_metrics_summary`, que resume la calidad por servicio, año y mes.

Con esto, el pipeline demuestra que no carga datos finales sin validación previa y que mantiene evidencia auditable de los registros rechazados y de las métricas de calidad obtenidas.